# Live SerialAnalysis signals
Run the install and setup cells, then run the live graph. Stop it before running the receive check; both use the same serial port.

The red dashed lines on the acceleration graph mark completed reps. The vitals view updates once per second.


In [ ]:
%pip install pyserial matplotlib

In [ ]:
from pathlib import Path
import sys
import time

# Allow launch from the repository root or the Live folder.
cwd = Path.cwd().resolve()
repo = cwd if (cwd / "SerialAnalysis" / "Live").is_dir() else cwd.parents[1]
if not (repo / "SerialAnalysis" / "Live").is_dir():
    raise RuntimeError("Start Jupyter from the repository root or SerialAnalysis/Live")
sys.path.insert(0, str(repo))

from IPython.display import display
from matplotlib import pyplot as plt
# Reload local code when rerunning setup after an edit.
import importlib
# Reload analysis first so recorder/live_plot pick up its latest symbols.
importlib.reload(importlib.import_module("SerialAnalysis.Live.analysis"))
recorder_module = importlib.reload(importlib.import_module("SerialAnalysis.Live.recorder"))
plot_module = importlib.reload(importlib.import_module("SerialAnalysis.Live.live_plot"))
LiveRecorder = recorder_module.LiveRecorder
LivePlot = plot_module.LivePlot
VitalsView = plot_module.VitalsView
run_jupyter_live_plot = plot_module.run_jupyter_live_plot
print(f"Loaded recorder from {recorder_module.__file__}")

PORT = "COM8"
OUTPUT = repo / "sessions"
DURATION_MIN = 30
WINDOW_S = 5
ECG_WINDOW_S = 1.0
# Target time between frames. A frame now costs ~25 ms, so 0.1 s (10 fps) is easy;
# lower it (e.g. 0.05) for a smoother scroll.
REFRESH_S = 0.1
# How many seconds of BPM/SpO2 history the vitals trend plot shows.
VITALS_WINDOW_S = 60


In [ ]:
if "recorder" in globals() and recorder is not None:
    recorder.close()
recorder = LiveRecorder(PORT, OUTPUT, DURATION_MIN, window_s=WINDOW_S,
                        vitals_window_s=VITALS_WINDOW_S)

plot = LivePlot(
    channels=["red", "infrared", "accel_magnitude",
              "diff_filtered", "wilson_filtered", "diff_raw", "wilson_raw"],
    window_s=WINDOW_S,
    channel_windows={name: ECG_WINDOW_S for name in
                     ("diff_filtered", "wilson_filtered", "diff_raw", "wilson_raw")},
    figsize=(10.0, 10.0),
    dpi=80,
)
# BPM, SpO2, and rep count readout plus a rolling heart-rate / SpO2 trend plot.
vitals = VitalsView(window_s=VITALS_WINDOW_S, figsize=(10.0, 3.0), dpi=80)

# The views draw to their own Agg canvases and are shown as images, which is
# much faster than re-rendering a matplotlib figure on every update.
vitals_handle = display(vitals.image(), display_id=True)
handle = display(plot.image(), display_id=True)

with recorder:
    run_jupyter_live_plot(recorder, plot, handle, refresh_s=REFRESH_S,
                          vitals_view=vitals, vitals_handle=vitals_handle)


In [ ]:
# Diagnostic: print incoming binary chunks and decoded frames for 10 seconds.
# Run this cell while the live graph cell is stopped.
from SerialAnalysis.Live.recorder import open_serial
from SerialAnalysis.Live.protocol import Decoder

MONITOR_SECONDS = 10
decoder = Decoder()
port = open_serial(PORT)
start = last_wait = time.monotonic()
received_bytes = received_frames = 0
print(f"Listening on {PORT} at {port.baudrate} baud, DTR={port.dtr}, for {MONITOR_SECONDS} seconds...", flush=True)
try:
    while time.monotonic() - start < MONITOR_SECONDS:
        chunk = port.read(1024)
        if chunk:
            received_bytes += len(chunk)
            print(f"RX {len(chunk)} bytes: {chunk.hex(' ')}", flush=True)
            frames = decoder.feed(chunk)
            received_frames += len(frames)
            for frame in frames:
                print(f"  frame type={frame.type} seq={frame.sequence} "
                      f"board_us={frame.timestamp_us} samples={frame.count}", flush=True)
        elif time.monotonic() - last_wait >= 1:
            print("Waiting for serial bytes...", flush=True)
            last_wait = time.monotonic()
except KeyboardInterrupt:
    print("Receive check interrupted.", flush=True)
finally:
    port.close()
    print(f"Closed {PORT}: {received_bytes} bytes, {received_frames} frames, "
          f"{decoder.bad} bad CRC/length frames.", flush=True)
